# Single-Cell RNA-seq Analysis

**Objective:** End-to-end scRNA-seq analysis on a synthetic 500-cell × 2000-gene
count matrix — QC, normalisation, HVG selection, PCA, 2D embedding, KMeans
clustering — with results written to a Snowflake table.

**Note:** No real scRNA tables are required; all data are generated in-notebook.

**Library dependencies (pre-installed in Workspaces):** numpy · pandas · sklearn · matplotlib  
**Optional:** umap-learn *(PCA 2D fallback if absent — notebook always runs)*

In [ ]:
# Uncomment to install optional UMAP library:
# %pip install umap-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

try:
    import umap as umap_lib
    UMAP_AVAILABLE = True
    print('umap-learn available — will use UMAP embedding')
except ImportError:
    UMAP_AVAILABLE = False
    print('umap-learn not installed — PCA 2D projection will be used (UMAP placeholder)')

from workbench_helpers import Workbench
wb      = Workbench(database='SCIENTIFIC_WORKBENCH', warehouse='WORKBENCH_S')
session = wb.session
print(f'Connected as: {session.sql("SELECT CURRENT_USER()").collect()[0][0]}')

In [ ]:
RANDOM_SEED   = 42
N_CELLS       = 500
N_GENES       = 2000
N_CLUSTERS    = 5
N_MT_GENES    = 50
N_HVG         = 500
N_PCA_COMPS   = 50
N_PCA_EMBED   = 30

RESULTS_TABLE = 'WORKBENCH_PROJECTS.SHARED_ANALYTICS.SCRNA_CELL_EMBEDDINGS'

print(f'Experiment : {N_CELLS} cells x {N_GENES} genes, {N_CLUSTERS} clusters')
print(f'Results    : {RESULTS_TABLE}')

---
## Phase 1 — Synthetic Count Matrix Generation

A Poisson-based generative model creates a count matrix with:
- **5 biologically distinct clusters** (marker gene programmes per cluster)
- **Mitochondrial genes** (first 50 features, labelled `MT-*`)
- **Deterministic** — seeded with `RANDOM_SEED = 42`

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

gene_names = [f'MT-GENE_{i:02d}' for i in range(N_MT_GENES)] + \
             [f'GENE_{i:04d}' for i in range(N_MT_GENES, N_GENES)]
cell_ids   = [f'CELL_{i:04d}' for i in range(N_CELLS)]

# Balanced cluster assignment
cell_clusters = np.tile(np.arange(N_CLUSTERS), N_CELLS // N_CLUSTERS)
if N_CELLS % N_CLUSTERS:
    cell_clusters = np.append(cell_clusters, np.arange(N_CELLS % N_CLUSTERS))
rng.shuffle(cell_clusters)

# Per-cluster mean expression programmes in log space
cluster_means = rng.normal(0.0, 1.5, (N_CLUSTERS, N_GENES))
for k in range(N_CLUSTERS):   # marker genes make clusters separable
    cluster_means[k, k*100 : k*100 + 80] += 3.0

# Poisson count matrix
log_lambda = cluster_means[cell_clusters] + rng.normal(0, 0.4, (N_CELLS, N_GENES))
rates      = np.exp(log_lambda) + 0.05
counts     = rng.poisson(rates).astype(np.float32)

print(f'Count matrix shape   : {counts.shape}')
print(f'Total UMIs           : {counts.sum():,.0f}')
print(f'Median UMIs / cell   : {np.median(counts.sum(axis=1)):.0f}')
print(f'Sparsity             : {(counts == 0).mean():.1%} zeros')
print(f'Cluster sizes        : {np.bincount(cell_clusters)}')

---
## Phase 2 — Quality Control

In [ ]:
lib_size = counts.sum(axis=1)
n_genes  = (counts > 0).sum(axis=1)
mt_frac  = counts[:, :N_MT_GENES].sum(axis=1) / (lib_size + 1e-9)

lib_lo, lib_hi = np.percentile(lib_size, [2.5, 97.5])
mt_hi          = 0.25
qc_pass = (lib_size >= lib_lo) & (lib_size <= lib_hi) & (mt_frac <= mt_hi)
print(f'QC: {qc_pass.sum()} / {N_CELLS} cells pass')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('QC Metrics', fontsize=12, fontweight='bold')
for ax, data, lbl, thr in zip(
    axes,
    [lib_size, n_genes, mt_frac * 100],
    ['Library size (UMIs)', 'Genes detected', 'MT fraction (%)'],
    [None, None, mt_hi * 100],
):
    ax.hist(data, bins=40, color='#6366f1', edgecolor='white', alpha=0.8)
    ax.set_xlabel(lbl); ax.set_ylabel('Cells')
    if thr is not None:
        ax.axvline(thr, color='crimson', ls='--', lw=1.5, label=f'threshold {thr:.1f}')
        ax.legend()
plt.tight_layout(); plt.show()

counts_qc  = counts[qc_pass]
cell_ids_qc = [cell_ids[i] for i in np.where(qc_pass)[0]]
clusters_qc = cell_clusters[qc_pass]
print(f'Post-QC matrix: {counts_qc.shape}')

---
## Phase 3 — Normalisation & Highly Variable Gene Selection

In [ ]:
# CP10K log-normalisation
lib_qc  = counts_qc.sum(axis=1, keepdims=True)
norm    = counts_qc / lib_qc * 1e4
log1p   = np.log1p(norm)

# HVG: select by dispersion (var / mean)
mean_g = log1p.mean(axis=0)
var_g  = log1p.var(axis=0)
disp   = var_g / (mean_g + 1e-9)

hvg_idx   = np.argsort(disp)[-N_HVG:]
hvg_names = [gene_names[i] for i in hvg_idx]
X_hvg     = log1p[:, hvg_idx]
print(f'HVG matrix: {X_hvg.shape} (cells x HVGs)')

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(mean_g, disp, s=3, alpha=0.3, c='#94a3b8', label='All genes')
ax.scatter(mean_g[hvg_idx], disp[hvg_idx], s=6, alpha=0.6, c='#ef4444',
           label=f'HVGs (n={N_HVG})')
ax.set_xlabel('Mean expression (log1p)'); ax.set_ylabel('Dispersion')
ax.set_title('HVG Selection'); ax.legend()
plt.tight_layout(); plt.show()

---
## Phase 4 — PCA

In [ ]:
scaler  = StandardScaler()
X_sc    = scaler.fit_transform(X_hvg)

pca     = PCA(n_components=N_PCA_COMPS, random_state=RANDOM_SEED)
X_pca   = pca.fit_transform(X_sc)
var_exp = pca.explained_variance_ratio_

print(f'PCA shape           : {X_pca.shape}')
print(f'Cumulative var (50) : {var_exp.sum():.1%}')

PALETTE = plt.cm.tab10(np.linspace(0, 0.5, N_CLUSTERS))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 21), var_exp[:20]*100, color='#6366f1')
axes[0].set_xlabel('PC'); axes[0].set_ylabel('% Variance Explained')
axes[0].set_title('Scree Plot')

for k in range(N_CLUSTERS):
    m = clusters_qc == k
    axes[1].scatter(X_pca[m,0], X_pca[m,1], s=8, alpha=0.6,
                    color=PALETTE[k], label=f'Cluster {k}')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA — PC1 vs PC2 (true labels)'); axes[1].legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.show()

---
## Phase 5 — 2D Embedding

Uses UMAP if `umap-learn` is installed; otherwise falls back to PCA 2D
(labelled *PCA (UMAP placeholder)*) so the notebook always runs end-to-end.

In [ ]:
X_embed = X_pca[:, :N_PCA_EMBED]

if UMAP_AVAILABLE:
    reducer     = umap_lib.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                                random_state=RANDOM_SEED)
    X_2d        = reducer.fit_transform(X_embed)
    embed_label = 'UMAP'
    print(f'UMAP embedding: {X_2d.shape}')
else:
    X_2d        = X_pca[:, :2]
    embed_label = 'PCA (UMAP placeholder)'
    print(f'PCA 2D fallback: {X_2d.shape}')

print(f'Embedding method: {embed_label}')

---
## Phase 6 — Clustering (KMeans as Leiden Substitute)

In [ ]:
# KMeans on top-30 PCs (Leiden substitute — noted in output)
km      = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10)
km_lbls = km.fit_predict(X_embed)
print(f'KMeans cluster sizes: {np.bincount(km_lbls)}')
print('(Note: KMeans on PCA top PCs is used as a Leiden substitute.)')
print('  Install leidenalg + igraph for graph-based clustering.')

PALETTE = plt.cm.tab10(np.linspace(0, 0.5, N_CLUSTERS))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ax0lbl = embed_label.split()[0]

for ax, labels, title in zip(
    axes,
    [clusters_qc, km_lbls],
    [f'{embed_label} — True Labels', f'{embed_label} — KMeans Labels'],
):
    for k in range(N_CLUSTERS):
        m = labels == k
        ax.scatter(X_2d[m,0], X_2d[m,1], s=8, alpha=0.65,
                   color=PALETTE[k], label=f'Cluster {k}')
    ax.set_xlabel(f'{ax0lbl} 1'); ax.set_ylabel(f'{ax0lbl} 2')
    ax.set_title(title); ax.legend(markerscale=2, fontsize=8)

plt.tight_layout(); plt.show()

---
## Phase 7 — Save Results to Snowflake

In [ ]:
results = pd.DataFrame({
    'CELL_ID':          cell_ids_qc,
    'TRUE_CLUSTER':     clusters_qc.astype(int),
    'KMEANS_CLUSTER':   km_lbls.astype(int),
    'PC1':              X_pca[:, 0].astype(float),
    'PC2':              X_pca[:, 1].astype(float),
    'EMBED_X':          X_2d[:, 0].astype(float),
    'EMBED_Y':          X_2d[:, 1].astype(float),
    'LIB_SIZE':         lib_qc.flatten().astype(int),
    'N_GENES_DETECTED': (counts_qc > 0).sum(axis=1).astype(int),
    'MT_FRACTION':      mt_frac[qc_pass].astype(float),
    'EMBED_METHOD':     embed_label,
})

# Write to Snowflake via Snowpark
snow_df = session.create_dataframe(results)
snow_df.write.mode('overwrite').save_as_table(RESULTS_TABLE)
print(f'Wrote {len(results)} rows to {RESULTS_TABLE}')

# Verify
n_written = session.sql(f'SELECT COUNT(*) FROM {RESULTS_TABLE}').collect()[0][0]
print(f'SELECT COUNT(*) -> {n_written} rows')
assert n_written == len(results), f'Row count mismatch: {n_written} != {len(results)}'
print('Row count verified OK.')

wb.log_decision(
    f'scRNA-seq analysis: {N_CELLS} cells x {N_GENES} genes synthetic matrix. '
    f'{qc_pass.sum()} cells passed QC. Embedding: {embed_label}. '
    f'Results: {RESULTS_TABLE}.',
    evidence=f'numpy Poisson simulation seed={RANDOM_SEED}; '
             f'sklearn PCA n={N_PCA_COMPS}; KMeans n={N_CLUSTERS}'
)
print('Provenance logged.')
print('\nNext steps:')
print('  - Replace synthetic data with real 10x Genomics cellranger output')
print('  - Install umap-learn for true UMAP:  %pip install umap-learn')
print('  - Connect RESULTS_TABLE to a Streamlit scRNA explorer dashboard')